In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "slovak-nlp/Qwen3-14B-sk"

print(f"Loading {model_name}...")

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)
model.eval()

# Qwen3 supports thinking mode — use /no_think for direct generation
# Use chat template for instruction-style prompts

def generate(prompt, max_new_tokens=200, use_thinking=False):
    suffix = "" if use_thinking else " /no_think"
    messages = [{"role": "user", "content": prompt + suffix}]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id,
        )

    # Decode only the newly generated tokens
    new_tokens = output_ids[0][len(inputs["input_ids"][0]):]
    response = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    return response


# Example 1 — voľné generovanie textu (bez thinking módu)
prompt1 = "Napíš krátky fakt o histórii Bratislavy."
print("\n--- Príklad 1: Generovanie textu ---")
print(f"Prompt: {prompt1}")
print(f"Odpoveď: {generate(prompt1)}")

# Example 2 — odpovedanie na otázku
prompt2 = "Čo je hlavné mesto Slovenska a koľko má obyvateľov?"
print("\n--- Príklad 2: Otázka a odpoveď ---")
print(f"Prompt: {prompt2}")
print(f"Odpoveď: {generate(prompt2)}")

# Example 3 — s thinking módom (pre komplexnejšie úlohy)
prompt3 = "Aké sú výhody a nevýhody jadrových elektrární?"
print("\n--- Príklad 3: S thinking módom ---")
print(f"Prompt: {prompt3}")
print(f"Odpoveď: {generate(prompt3, use_thinking=True)}")

Loading slovak-nlp/Qwen3-14B-sk...


Loading weights:   0%|          | 0/443 [00:00<?, ?it/s]


--- Príklad 1: Generovanie textu ---
Prompt: Napíš krátky fakt o histórii Bratislavy.
Odpoveď: <think>

</think>

</think>

V roku 1993 sa Bratislava stala hlavným mestom Slovenskej republiky.

--- Príklad 2: Otázka a odpoveď ---
Prompt: Čo je hlavné mesto Slovenska a koľko má obyvateľov?
Odpoveď: <think>

</think>

<think>

Bratislava, 425 000 obyvateľov

--- Príklad 3: S thinking módom ---
Prompt: Aké sú výhody a nevýhody jadrových elektrární?
Odpoveď: <think>

</think>

Výhodou je nezávislosť na dodávkach fosílnych palív a možnosť výroby elektrickej energie bez vypúšťania emisií skleníkových plynov. Nevýhodou je vysoké investičné zaťaženie a riziko nežiaducich jadrových reakcií.
